# Understanding nearest_neighbor_tsp() Output

This notebook demonstrates how to understand and work with the output from the `nearest_neighbor_tsp()` function.

**Related Documentation:** See `docs/understanding_nearest_neighbor_output.md` for detailed explanation.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph import load_default_graph
from src.solvers import nearest_neighbor_tsp
import time

# Load the Singapore MRT/LRT network
print("Loading graph...")
G = load_default_graph()
print(f"✓ Loaded {G.number_of_nodes()} stations, {G.number_of_edges()} connections")

## Running nearest_neighbor_tsp with BP10

Let's run the function and examine what it returns:

In [ ]:
# Run nearest neighbor starting from BP10 (Fajar station)
print("Running: nn_path = nearest_neighbor_tsp(G, start_station='BP10')")
print()

start_time = time.time()
nn_path = nearest_neighbor_tsp(G, start_station='BP10')
elapsed = time.time() - start_time

print(f"✓ Completed in {elapsed:.4f} seconds")

## Understanding the Output Structure

The return value `nn_path` is a **tuple** with 2 elements:

In [ ]:
print("=" * 70)
print("OUTPUT STRUCTURE")
print("=" * 70)

print(f"\nnn_path is a: {type(nn_path)}")
print(f"Number of elements: {len(nn_path)}")
print()

print(f"Element 0 (nn_path[0]): {type(nn_path[0]).__name__}")
print(f"Element 1 (nn_path[1]): {type(nn_path[1]).__name__}")
print()

print("💡 Best practice: Unpack immediately")
print("   tour, cost = nearest_neighbor_tsp(G, start_station='BP10')")

## Element 1: The Tour (nn_path[0])

The **tour** is a list of station IDs in visit order:

In [ ]:
tour = nn_path[0]  # Extract the tour

print("=" * 70)
print("THE TOUR (nn_path[0])")
print("=" * 70)

print(f"\nType: {type(tour)}")
print(f"Length: {len(tour)} stations")
print(f"Starting station: {tour[0]}")
print(f"Ending station: {tour[-1]} (then returns to {tour[0]})")
print()

print("First 10 stations in the tour:")
for i in range(min(10, len(tour))):
    station_id = tour[i]
    station_name = G.nodes[station_id]['name']
    line_code = G.nodes[station_id]['line_code']
    print(f"  {i+1:3d}. {station_id:5s} ({line_code} Line) - {station_name}")

print()
print("Last 10 stations in the tour:")
for i in range(max(0, len(tour)-10), len(tour)):
    station_id = tour[i]
    station_name = G.nodes[station_id]['name']
    line_code = G.nodes[station_id]['line_code']
    print(f"  {i+1:3d}. {station_id:5s} ({line_code} Line) - {station_name}")

## Element 2: The Total Cost (nn_path[1])

The **cost** is the total travel time for the complete tour:

In [ ]:
cost = nn_path[1]  # Extract the cost

print("=" * 70)
print("THE COST (nn_path[1])")
print("=" * 70)

print(f"\nType: {type(cost)}")
print(f"Value: {cost:.2f} minutes")
print()

print("Interpreting the cost:")
print(f"  ⏱️  Total travel time: {cost:.2f} minutes")
print(f"  ⏱️  Total travel time: {cost/60:.2f} hours")
print(f"  📊 Average per station: {cost/len(tour):.2f} minutes")
print()

print("What this represents:")
print(f"  The time to travel: {tour[0]} → {tour[1]} → ... → {tour[-1]} → {tour[0]}")
print(f"  (visiting all {len(tour)} stations and returning to start)")

## Proper Unpacking Pattern

The recommended way to use the function:

In [ ]:
# ✅ RECOMMENDED: Unpack immediately
tour, cost = nearest_neighbor_tsp(G, start_station='BP10')

print("✅ Recommended usage:")
print("   tour, cost = nearest_neighbor_tsp(G, start_station='BP10')")
print()
print(f"tour: list of {len(tour)} stations")
print(f"cost: {cost:.2f} minutes")
print()

# Now you can work with tour and cost directly
print("Examples of what you can do:")
print(f"  - Check tour length: len(tour) = {len(tour)}")
print(f"  - Get first station: tour[0] = '{tour[0]}'")
print(f"  - Get last station: tour[-1] = '{tour[-1]}'")
print(f"  - Calculate hours: cost/60 = {cost/60:.2f}")
print(f"  - Average per station: cost/len(tour) = {cost/len(tour):.2f}")

## Validating the Tour

Let's verify that the tour is valid:

In [ ]:
from src.utils import validate_tour

print("=" * 70)
print("TOUR VALIDATION")
print("=" * 70)

# Check basic properties
print(f"\n✓ All stations visited: {len(tour) == G.number_of_nodes()}")
print(f"✓ No duplicates: {len(tour) == len(set(tour))}")
print(f"✓ Starts at BP10: {tour[0] == 'BP10'}")

# Check that all stations exist in graph
all_valid = all(station in G.nodes() for station in tour)
print(f"✓ All stations valid: {all_valid}")

# Use the validation function (it will raise an error if invalid)
try:
    from src.utils.metric import build_metric_closure
    closure = build_metric_closure(G)
    validate_tour(tour, closure, require_complete=True)
    print("✓ Tour validation passed")
except ValueError as e:
    print(f"✗ Tour validation failed: {e}")

## Working with Individual Stations

Let's examine some stations in detail:

In [ ]:
print("=" * 70)
print("STATION DETAILS")
print("=" * 70)

# Starting station (BP10)
start_id = tour[0]
print(f"\nStarting Station: {start_id}")
for key, value in G.nodes[start_id].items():
    print(f"  {key:20s}: {value}")

# Second station
second_id = tour[1]
print(f"\nSecond Station: {second_id}")
for key, value in G.nodes[second_id].items():
    print(f"  {key:20s}: {value}")

# Last station before returning
last_id = tour[-1]
print(f"\nLast Station (before returning to start): {last_id}")
for key, value in G.nodes[last_id].items():
    print(f"  {key:20s}: {value}")

## Summary Statistics

In [ ]:
print("=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"\n📍 Starting Station: {tour[0]} ({G.nodes[tour[0]]['name']})")
print(f"📊 Total Stations: {len(tour)}")
print(f"⏱️  Total Time: {cost:.2f} minutes ({cost/60:.2f} hours)")
print(f"📈 Average per Station: {cost/len(tour):.2f} minutes")
print(f"⚡ Computation Time: {elapsed:.4f} seconds")
print()

print("Understanding the result:")
print("  • tour (nn_path[0]): List of station IDs in visit order")
print("  • cost (nn_path[1]): Total travel time in minutes")
print("  • The tour forms a cycle (returns to starting station)")
print("  • Uses shortest paths between stations (metric closure)")
print()

print("Next steps:")
print("  1. Try different starting stations to compare results")
print("  2. Improve with 2-opt: improve_tour_2opt(tour, G)")
print("  3. Try metaheuristics: simulated_annealing_tsp(G)")
print("  4. Use multi-start: nearest_neighbor_multi_start(G, num_starts=10)")

## Additional Examples

### Example 1: Try Different Starting Stations

In [ ]:
print("Comparing different starting stations:\n")

# Try a few different starting points
test_starts = ['BP10', 'NS1', 'EW1', 'NE1', 'CC1']

results = []
for start in test_starts:
    tour, cost = nearest_neighbor_tsp(G, start_station=start)
    station_name = G.nodes[start]['name']
    results.append((start, station_name, cost))

# Sort by cost
results.sort(key=lambda x: x[2])

print(f"{'Rank':<6} {'Station':<8} {'Name':<25} {'Cost (min)':>12} {'Cost (hr)':>10}")
print("-" * 70)
for i, (station_id, name, cost) in enumerate(results, 1):
    print(f"{i:<6} {station_id:<8} {name:<25} {cost:>12.2f} {cost/60:>10.2f}")

best = results[0]
worst = results[-1]
diff = worst[2] - best[2]
pct_diff = (diff / best[2]) * 100

print(f"\n💡 Starting point matters! Difference between best and worst: {diff:.2f} min ({pct_diff:.1f}%)")

### Example 2: Using Multi-Start for Better Results

In [ ]:
from src.solvers import nearest_neighbor_multi_start

print("Running Nearest Neighbor with multiple starting points...\n")

# Try 20 different starting points
best_tour, best_cost, best_start = nearest_neighbor_multi_start(G, num_starts=20)

print(f"Best result from 20 starting points:")
print(f"  Starting station: {best_start} ({G.nodes[best_start]['name']})")
print(f"  Tour cost: {best_cost:.2f} minutes ({best_cost/60:.2f} hours)")
print(f"  Average per station: {best_cost/len(best_tour):.2f} minutes")

# Compare to single start from BP10
bp10_tour, bp10_cost = nearest_neighbor_tsp(G, start_station='BP10')
improvement = bp10_cost - best_cost
pct_improvement = (improvement / bp10_cost) * 100

print(f"\nComparison to single start (BP10):")
print(f"  BP10 cost: {bp10_cost:.2f} minutes")
print(f"  Multi-start best: {best_cost:.2f} minutes")
print(f"  Improvement: {improvement:.2f} minutes ({pct_improvement:.2f}%)")

### Example 3: Improving with 2-opt

In [ ]:
from src.solvers import improve_tour_2opt

print("Improving the BP10 tour with 2-opt optimization...\n")

# Start with BP10 tour
initial_tour, initial_cost = nearest_neighbor_tsp(G, start_station='BP10')

# Improve with 2-opt
start_time = time.time()
improved_tour, _, improved_cost = improve_tour_2opt(initial_tour, G, verbose=False)
opt_time = time.time() - start_time

improvement = initial_cost - improved_cost
pct_improvement = (improvement / initial_cost) * 100

print(f"Results:")
print(f"  Initial (NN): {initial_cost:.2f} minutes")
print(f"  After 2-opt: {improved_cost:.2f} minutes")
print(f"  Improvement: {improvement:.2f} minutes ({pct_improvement:.2f}%)")
print(f"  Optimization time: {opt_time:.4f} seconds")
print()
print(f"💡 2-opt found a {pct_improvement:.2f}% better solution!")

## Key Takeaways

1. **Return Value:** `nearest_neighbor_tsp()` returns a tuple `(tour, cost)`
   - `tour`: List of station IDs in visit order
   - `cost`: Total travel time in minutes

2. **Always Unpack:** Use `tour, cost = nearest_neighbor_tsp(...)` for clarity

3. **Starting Point Matters:** Different starting stations give different results
   - Use `nearest_neighbor_multi_start()` to try multiple starting points

4. **It's a Heuristic:** Nearest Neighbor gives good but not optimal solutions
   - Improve with 2-opt local search
   - Or use metaheuristics (Simulated Annealing, Genetic Algorithm)

5. **Metric Closure:** The algorithm uses shortest paths between stations
   - Not direct connections only
   - Ensures every station can reach every other station

For more details, see: `docs/understanding_nearest_neighbor_output.md`